# Advanced Analytics — Bluestock Mutual Fund Platform

Risk, rolling Sharpe, investor cohorts, SIP continuity, fund recommendations and sector concentration.

Annual risk-free rate: 6.5%. Rolling window: 90 observed trading-day returns.

## 1. Historical VaR and CVaR

VaR is the 5th percentile of observed daily NAV returns. CVaR is the mean of returns at or below that threshold. These are historical estimates, not forecasts.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Image
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
          PROJECT_DIR = PROJECT_DIR.parent
REPORT_DIR = PROJECT_DIR / 'reports'
var_df = pd.read_csv(REPORT_DIR / 'var_cvar_report.csv')
display(var_df.sort_values('VaR_95_pct').head(10))
display(var_df[['VaR_95_pct', 'CVaR_95_pct']].describe().round(3))

## 2. Rolling 90-Day Sharpe Ratio

The five schemes are selected by highest full-period Sharpe ratios in the supplied performance dataset. Rolling Sharpe uses 90 observed NAV return rows and the 6.5% annual risk-free rate.

In [ ]:
selected = pd.read_csv(REPORT_DIR / 'rolling_sharpe_selected_funds.csv')
display(selected)
display(Image(filename=str(REPORT_DIR / 'rolling_sharpe_chart.png')))

## 3. Investor Cohort Analysis

Cohorts are based on each investor's first transaction year. Retention means transaction activity in a subsequent calendar year. Missing periods are unavailable observations, not zero retention.

In [ ]:
cohort = pd.read_csv(REPORT_DIR / 'cohort_analysis.csv')
display(cohort)

## 4. SIP Continuity

An investor is flagged only if they have at least six recorded SIP transactions and a maximum gap between consecutive recorded SIP transactions greater than 35 days. This is a historical screening rule.

In [ ]:
sip = pd.read_csv(REPORT_DIR / 'sip_continuity.csv')
eligible = sip[sip['sip_transaction_count'] >= 6]
at_risk = eligible[eligible['at_risk'].astype(bool)]
print('Investors analysed:', len(sip))
print('Eligible investors:', len(eligible))
print('At-risk investors:', len(at_risk))
print('At-risk share:', round(len(at_risk)/len(eligible)*100, 2) if len(eligible) else 0, '%')
display(at_risk.sort_values('maximum_gap_days', ascending=False).head(10))

## 5. Fund Recommendations by Risk Bucket

Low remains Low; Moderate remains Moderate; Moderately High, High and Very High are grouped into High. Funds are ranked by Sharpe ratio within each bucket. This is not personalised investment advice.

In [ ]:
recommendations = pd.read_csv(REPORT_DIR / 'fund_recommendations.csv')
for bucket in ['Low', 'Moderate', 'High']:
          print('\
' + bucket + ' Risk')
          display(recommendations[recommendations['risk_bucket'] == bucket])

## 6. Sector Concentration — HHI

HHI is the sum of squared normalised sector weights multiplied by 10,000. Only Equity schemes are included, using each fund's latest available holdings snapshot.

In [ ]:
hhi = pd.read_csv(REPORT_DIR / 'sector_hhi.csv')
display(hhi.sort_values('sector_hhi', ascending=False))
display(Image(filename=str(REPORT_DIR / 'sector_hhi_chart.png')))

## 7. Key Findings

In [ ]:
print((REPORT_DIR / 'advanced_key_insights.txt').read_text(encoding='utf-8'))

## 8. Methodology and Limitations

- VaR/CVaR are historical estimates, not forecasts.
- Rolling Sharpe uses 90 observed NAV return rows, not calendar days.
- Cohort retention measures later-year transaction activity, not assets retained.
- SIP gaps are measured between recorded transactions; contractual due dates are not available.
- Recommendations rank supplied funds by Sharpe within broad risk buckets.
- HHI measures sector concentration in the latest supplied Equity holdings snapshot.
- Results depend on supplied data coverage and are not personalised investment advice.